# Bitcoin VNMAX · render dos documentários dos países no Colab

Baixa o pacote do país dos releases públicos de `Changzaoo/vnmax-render`, renderiza e grava a saída **no Google Drive, capítulo por capítulo** (`VNMAX/saida/<ISO>/`). Se a sessão cair, abra de novo e rode todas as células: o que já está no Drive é pulado.

Edite só a 1ª célula (ISO) e use **Ambiente de execução › Executar tudo**.

In [ ]:
# ── parâmetros ──
ISO = "BR-c05"       # nome do pacote no release (ex.: BR, US, AR, BR-c02)
WORKERS = 0          # 0 = núcleos - 1 (mínimo 1)
CAPITULOS = ""       # "" = todos; ou "2,5"
ORIGEM = "https://github.com/Changzaoo/vnmax-render/releases/download/pacotes/"
DRIVE = "/content/drive/MyDrive/VNMAX"
BASE = "/content/vnmax-render"   # disco local da sessão (quadros temporários)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
for p in ("saida",):
    os.makedirs(os.path.join(DRIVE, p), exist_ok=True)
!nproc; free -g | head -2; df -h /content | tail -1


In [ ]:
# ── scripts e pacote, direto do release público ──
!mkdir -p /content/remoto "$BASE/pacotes"
!wget -q -O /content/remoto/bootstrap.sh "$ORIGEM"bootstrap.sh && wget -q -O /content/remoto/render_remoto.py "$ORIGEM"render_remoto.py
PACOTE = os.path.join(BASE, "pacotes", ISO + ".tar.gz")
if not os.path.exists(PACOTE):
    !wget -q -O "$PACOTE" "$ORIGEM$ISO".tar.gz
print(PACOTE, round(os.path.getsize(PACOTE) / 1048576, 1), "MB")


In [ ]:
# ── instala node 22, Chrome headless, ffmpeg e hyperframes (idempotente; ~3-5 min na 1ª vez) ──
!VNMAX_BASE="$BASE" bash /content/remoto/bootstrap.sh 2>&1 | tail -25


In [ ]:
# ── render: cada capítulo pronto vai direto para o Drive ──
SAIDA = os.path.join(DRIVE, "saida", ISO)
cmd = ["python3", "-u", "/content/remoto/render_remoto.py", PACOTE, "--base", BASE, "--saida", SAIDA]
if WORKERS: cmd += ["--workers", str(WORKERS)]
if CAPITULOS: cmd += ["--capitulos", CAPITULOS]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in proc.stdout:
    print(linha, end="")
print("código de saída:", proc.wait())


In [ ]:
# ── onde parou + garante que o Drive gravou tudo ──
import os
p = os.path.join(SAIDA, "pronto.json")
print(open(p).read() if os.path.exists(p) else "ainda sem pronto.json")
drive.flush_and_unmount()
print("Drive gravado. Se faltam capítulos, rode de novo a partir da 2ª célula.")
